# TREC-COVID — BM25 + ColBERT + RRFRelevance-preserving subset: all qrel docs + controlled non-relevant padding.Subset size kept small enough for Colab T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid"
TOP_K = 100          # retrieval depth — enough for Recall@100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000  # non-relevant padding docs

Mounted at /content/drive


In [ ]:
!pip install -q "numpy==1.26.4" --force-reinstall
!pip install -q beir pytrec-eval-terrier

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.

⚠️ **After Cell above runs → Runtime → Restart session → then continue from next cell**

In [21]:
%%time
from beir.datasets.data_loader import GenericDataLoader
import re, unicodedata, random

DATA_PATH = "/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid"
TOP_K = 100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000

corpus, queries, qrels = GenericDataLoader(data_folder=DATA_PATH).load(split="test")
print(f"Full corpus: {len(corpus):,} docs")
print(f"Queries: {len(queries):,}")
print(f"Qrels: {len(qrels):,}")

def shared_clean(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for doc_id, doc in corpus.items():
    doc["title"] = shared_clean(doc.get("title", ""))
    doc["text"]  = shared_clean(doc.get("text", ""))

for q_id, q in queries.items():
    queries[q_id] = shared_clean(q)

# --- Building relevance-preserving subset ---
relevant_doc_ids = set()
for q_id, docs in qrels.items():
    relevant_doc_ids.update(docs.keys())

corpus_subset = {did: corpus[did] for did in relevant_doc_ids if did in corpus}
n_relevant = len(corpus_subset)
print(f"Relevant docs in corpus: {n_relevant:,}")

non_relevant_ids = [did for did in corpus if did not in relevant_doc_ids]
random.seed(RANDOM_SEED)
pad_ids = random.sample(non_relevant_ids, min(NON_RELEVANT_PAD, len(non_relevant_ids)))
for did in pad_ids:
    corpus_subset[did] = corpus[did]

subset_doc_ids = list(corpus_subset.keys())
SUBSET_SIZE = len(corpus_subset)
print(f"Final subset: {SUBSET_SIZE:,} docs ({n_relevant:,} relevant + {SUBSET_SIZE - n_relevant:,} non-relevant)")

  0%|          | 0/171332 [00:00<?, ?it/s]

Full corpus: 171,332 docs
Queries: 50
Qrels: 50
Relevant docs in corpus: 35,480
Final subset: 70,480 docs (35,480 relevant + 35,000 non-relevant)
CPU times: user 14.5 s, sys: 424 ms, total: 14.9 s
Wall time: 16.2 s


## BM25 Retrieval

In [ ]:
!pip install -q elasticsearch==7.9.1
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!sudo chown -R daemon:daemon elasticsearch-7.9.2/
!shasum -a 512 -c elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512

elasticsearch-oss-7.9.2-linux-x86_64.tar.gz: OK


In [23]:
%%bash --bg
sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch

In [24]:
import time
time.sleep(30)
!curl -s http://localhost:9200 | python3 -c 'import sys, json; d=json.load(sys.stdin); print(f"ES {d[\"version\"][\"number\"]} is up")'

  File "<string>", line 1
    import sys, json; d=json.load(sys.stdin); print(f"ES {d[\"version\"][\"number\"]} is up")
                                                             ^
SyntaxError: unexpected character after line continuation character


In [25]:
!curl -s http://localhost:9200

{
  "name" : "e019c24d3950",
  "cluster_name" : "elasticsearch",
  "cluster_uuid" : "U1gPbSaZSBK6LuYnBmeIWQ",
  "version" : {
    "number" : "7.9.2",
    "build_flavor" : "oss",
    "build_type" : "tar",
    "build_hash" : "d34da0ea4a966c4e49417f2da2f244e3e97b4e6e",
    "build_date" : "2020-09-23T00:45:33.626720Z",
    "build_snapshot" : false,
    "lucene_version" : "8.6.2",
    "minimum_wire_compatibility_version" : "6.8.0",
    "minimum_index_compatibility_version" : "6.0.0-beta1"
  },
  "tagline" : "You Know, for Search"
}


In [ ]:
!pip uninstall -y numpy
!pip install -q numpy==1.26.4 elasticsearch==7.9.1 beir pytrec-eval-terrier

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", bu

In [26]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")
assert es.ping(), "ES not reachable!"

bm25_index_name = "trec-covid_bm25"

bm25_index_config = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "similarity": {
            "default": {
                "type": "BM25",
                "k1": 0.9,
                "b": 0.4
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

if es.indices.exists(index=bm25_index_name):
    es.indices.delete(index=bm25_index_name)
    print("Old index deleted")

es.indices.create(index=bm25_index_name, body=bm25_index_config)
print("Index created")

# Bulk index
def generate_bm25_docs():
    for doc_id, doc in corpus_subset.items():
        yield {
            "_index": bm25_index_name,
            "_id": doc_id,
            "_source": {
                "title": doc.get("title", ""),
                "text": doc.get("text", "")
            }
        }

success, failed = bulk(
    es,
    generate_bm25_docs(),
    chunk_size=500,
    request_timeout=300,
    stats_only=True,
    refresh='wait_for'   # important
)
print(f"Indexed: {success:,} success, {failed} failed")

es.indices.refresh(index=bm25_index_name)

count = es.count(index=bm25_index_name)["count"]
print(f"Indexed count after refresh: {count:,}")

assert count == success, f"Expected {success}, got {count}"
print(f"Verified: {count:,} docs in index")

Old index deleted
Index created
Indexed: 70,480 success, 0 failed
Indexed count after refresh: 70,480
Verified: 70,480 docs in index


In [ ]:
sample_q = list(queries.values())[0]
res = es.search(index=bm25_index_name, body={"query": {"multi_match": {"query": sample_q, "fields": ["title","text"]}}, "size": 3})
print(f"Query: {sample_q}")
for hit in res["hits"]["hits"]:
    print(f"  [{hit['_score']:.2f}] {hit['_source']['title'][:80]}")

Query: what is the origin of COVID-19
  [16.99] [What is the origin of SARS-CoV-2?]
  [12.88] Emergence of COVID-19 Infection: What Is Known and What Is to Be Expected-Narrat
  [12.87] EEG during the COVID-19 pandemic: What remains the same and what is different


In [ ]:
%%time
from tqdm import tqdm

bm25_results = {}
for q_id, q_text in tqdm(queries.items(), desc="BM25"):
    try:
        res = es.search(
            index=bm25_index_name,
            body={"query": {"multi_match": {"query": q_text, "fields": ["title","text"]}}, "size": TOP_K},
            request_timeout=30
        )
        bm25_results[q_id] = {hit["_id"]: float(hit["_score"]) for hit in res["hits"]["hits"]}
    except Exception as e:
        print(f"Error {q_id}: {e}")
        bm25_results[q_id] = {}

print(f"BM25 retrieval done: {len(bm25_results):,} queries")

BM25: 100%|██████████| 50/50 [00:02<00:00, 20.34it/s]

BM25 retrieval done: 50 queries
CPU times: user 124 ms, sys: 5.9 ms, total: 130 ms
Wall time: 2.46 s


In [ ]:
sample_q = next(iter(bm25_eval))
print(bm25_eval[sample_q].keys())

dict_keys(['recip_rank', 'recall_100', 'ndcg_cut_10'])


In [ ]:
import pytrec_eval, pickle

qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
metrics = {"ndcg_cut.10", "recip_rank.10", "recall.100"}
evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, metrics)

bm25_eval = evaluator.evaluate(bm25_results)
bm25_ndcg   = [bm25_eval[q]["ndcg_cut_10"] for q in bm25_eval]
bm25_mrr    = [bm25_eval[q]["recip_rank"]   for q in bm25_eval]
bm25_recall = [bm25_eval[q]["recall_100"]   for q in bm25_eval]

print("=" * 50)
print("BM25 Results — trec-covid")
print("=" * 50)
print(f"nDCG@10:    {sum(bm25_ndcg)/len(bm25_ndcg):.4f}")
print(f"MRR:        {sum(bm25_mrr)/len(bm25_mrr):.4f}")
print(f"Recall@100: {sum(bm25_recall)/len(bm25_recall):.4f}")

with open("/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid/trec-covid_results.pkl", "wb") as f:
    pickle.dump(bm25_results, f)
print("BM25 results saved to Drive")

BM25 Results — trec-covid
nDCG@10:    0.4822
MRR:        0.6932
Recall@100: 0.0927
BM25 results saved to Drive


## ColBERT Retrieval⚠️ ColBERT section reinstalls packages. After Cell below runs, **Restart session**, then run from the ColBERT reload cell onward.

In [9]:
!pip uninstall -y numpy transformers ragatouille langchain langchain-core colbert-ai -q
!pip install -q numpy==1.26.4
!pip install -q transformers==4.37.2
!pip install -q ragatouille==0.0.8
!pip install -q langchain==0.1.20 langchain-core==0.1.53
!pip install -q beir pytrec-eval-terrier
print("ColBERT packages installed — RESTART SESSION NOW, then run from next cell")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requir

⚠️ **Restart session now → then run from next cell**

In [1]:
import numpy as np
import torch
print(f"numpy: {np.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import transformers
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
transformers.AdamW = AdamW
transformers.get_linear_schedule_with_warmup = get_linear_schedule_with_warmup

from ragatouille import RAGPretrainedModel
print("RAGatouille loaded")

import time as time_module
import ragatouille.models.index as rag_index
rag_index.time = time_module
print("Patches applied")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


numpy: 1.26.4
CUDA: True
GPU: Tesla T4
RAGatouille loaded
Patches applied


In [2]:
%%time
from beir.datasets.data_loader import GenericDataLoader
import re, unicodedata, random, pickle

DATA_PATH = "/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid"
TOP_K = 100
RANDOM_SEED = 42
NON_RELEVANT_PAD = 35000

def shared_clean(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus, queries, qrels = GenericDataLoader(data_folder=DATA_PATH).load(split="test")

for doc_id, doc in corpus.items():
    doc["title"] = shared_clean(doc.get("title", ""))
    doc["text"] = shared_clean(doc.get("text", ""))
for q_id, q in queries.items():
    queries[q_id] = shared_clean(q)

# Rebuild EXACT same subset
relevant_doc_ids = set()
for q_id, docs in qrels.items():
    relevant_doc_ids.update(docs.keys())
corpus_subset = {did: corpus[did] for did in relevant_doc_ids if did in corpus}
non_relevant_ids = [did for did in corpus if did not in relevant_doc_ids]
random.seed(RANDOM_SEED)
pad_ids = random.sample(non_relevant_ids, min(NON_RELEVANT_PAD, len(non_relevant_ids)))
for did in pad_ids:
    corpus_subset[did] = corpus[did]
subset_doc_ids = list(corpus_subset.keys())
print(f"Subset rebuilt: {len(corpus_subset):,} docs")

# Load saved BM25 results
bm25_results = pickle.load(open("/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid/trec-covid_results.pkl","rb"))
print(f"BM25 results loaded: {len(bm25_results):,} queries")

# Build qrels + evaluator
qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
from pytrec_eval import RelevanceEvaluator
evaluator = RelevanceEvaluator(qrels_int, {"ndcg_cut.10", "recip_rank.10", "recall.100"})
print("Evaluator ready")

  0%|          | 0/171332 [00:00<?, ?it/s]

Subset rebuilt: 70,480 docs
BM25 results loaded: 50 queries
Evaluator ready
CPU times: user 13.8 s, sys: 370 ms, total: 14.1 s
Wall time: 15 s


In [3]:
colbert_documents = []
colbert_document_ids = []

for doc_id in subset_doc_ids:
    doc = corpus_subset[doc_id]
    full_text = f"{doc['title']} {doc['text']}".strip()
    colbert_documents.append(full_text)
    colbert_document_ids.append(str(doc_id))

print(f"ColBERT docs prepared: {len(colbert_documents):,}")

ColBERT docs prepared: 70,480


In [4]:
%%time
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

RAG.index(
    collection=colbert_documents,
    document_ids=colbert_document_ids,
    index_name="trec-covid_colbert",
    max_document_length=180,
    split_documents=False,
    use_faiss=True
)
print("ColBERT indexing complete")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


artifact.metadata: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

[Apr 08, 11:17:16] [0] 		 #> Encoding 20480 passages..


3it [05:07, 102.44s/it]
100%|██████████| 3/3 [00:00<00:00, 129.93it/s]


[Apr 08, 11:18:46] #> Optimizing IVF to store map from centroids to list of pids..
[Apr 08, 11:18:46] #> Building the emb2pid mapping..
[Apr 08, 11:18:46] len(emb2pid) = 8667906


100%|██████████| 32768/32768 [00:00<00:00, 46461.26it/s]


[Apr 08, 11:18:47] #> Saved optimized IVF to .ragatouille/colbert/indexes/trec-covid_colbert/ivf.pid.pt
Done indexing!
ColBERT indexing complete
CPU times: user 1h 52min 50s, sys: 18.9 s, total: 1h 53min 9s
Wall time: 1h 56min 11s


In [5]:
q = list(queries.values())[0]
print(f"Query: {q}")
for hit in RAG.search(q, k=5):
    print(f"  [{hit['score']:.4f}] {hit['document_id']}")

Query: what is the origin of COVID-19
Loading searcher for index trec-covid_colbert for the first time... This may take a few seconds


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Apr 08, 11:19:05] #> Loading codec...
[Apr 08, 11:19:05] #> Loading IVF...
[Apr 08, 11:19:05] #> Loading doclens...


/usr/local/lib/python3.12/dist-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|██████████| 3/3 [00:00<00:00, 900.00it/s]

[Apr 08, 11:19:05] #> Loading codes and residuals...



100%|██████████| 3/3 [00:00<00:00,  8.94it/s]


Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: . what is the origin of COVID-19, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2054,  2003,  1996,  4761,  1997,  2522, 17258,  1011,
         2539,   102,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

  [27.1406] 4dtk1kyh
  [26.5938] ptizke03
  [26.5781] utsr0zv7
  [26.5312] 1mjaycee
  [26.5000] sh7lrdou


In [7]:
%%time
from tqdm import tqdm

colbert_results = {}
for q_id, q_text in tqdm(queries.items(), desc="ColBERT"):
    hits = RAG.search(q_text, k=TOP_K)
    colbert_results[q_id] = {str(h["document_id"]): float(h["score"]) for h in hits}

print(f"ColBERT retrieval done: {len(colbert_results):,} queries")

ColBERT: 100%|██████████| 50/50 [00:04<00:00, 12.21it/s]

ColBERT retrieval done: 50 queries
CPU times: user 2.12 s, sys: 8.85 ms, total: 2.13 s
Wall time: 4.1 s


In [9]:
colbert_eval = evaluator.evaluate(colbert_results)
colbert_ndcg   = [colbert_eval[q]["ndcg_cut_10"] for q in colbert_eval]
colbert_mrr    = [colbert_eval[q]["recip_rank"]   for q in colbert_eval]
colbert_recall = [colbert_eval[q]["recall_100"]   for q in colbert_eval]

print("=" * 50)
print("ColBERT Results — trec-covid")
print("=" * 50)
print(f"nDCG@10:    {sum(colbert_ndcg)/len(colbert_ndcg):.4f}")
print(f"MRR:        {sum(colbert_mrr)/len(colbert_mrr):.4f}")
print(f"Recall@100: {sum(colbert_recall)/len(colbert_recall):.4f}")

import pickle
with open("/content/drive/MyDrive/colbert_results_trec-covid.pkl", "wb") as f:
    pickle.dump(colbert_results, f)
print("ColBERT results saved to Drive")

ColBERT Results — trec-covid
nDCG@10:    0.7303
MRR:        0.9162
Recall@100: 0.1371
ColBERT results saved to Drive


## RRF Fusion

In [10]:
from collections import defaultdict

def rrf_fuse(bm25_res, colbert_res, k=60):
    """Reciprocal Rank Fusion (Cormack et al., 2009).
    RRF(d) = sum_r 1/(k + rank_r(d)), k=60.
    """
    fused = {}
    for qid in set(bm25_res) | set(colbert_res):
        scores = defaultdict(float)
        for rank, (did, _) in enumerate(list(bm25_res.get(qid, {}).items()), 1):
            scores[did] += 1.0 / (k + rank)
        for rank, (did, _) in enumerate(list(colbert_res.get(qid, {}).items()), 1):
            scores[did] += 1.0 / (k + rank)
        fused[qid] = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))
    return fused

rrf_results = rrf_fuse(bm25_results, colbert_results, k=60)
print(f"RRF fusion: {len(rrf_results):,} queries")

RRF fusion: 50 queries


In [12]:
rrf_eval = evaluator.evaluate(rrf_results)
rrf_ndcg   = [rrf_eval[q]["ndcg_cut_10"] for q in rrf_eval]
rrf_mrr    = [rrf_eval[q]["recip_rank"]   for q in rrf_eval]
rrf_recall = [rrf_eval[q]["recall_100"]   for q in rrf_eval]

print("=" * 50)
print("RRF Hybrid Results — trec-covid")
print("=" * 50)
print(f"nDCG@10:    {sum(rrf_ndcg)/len(rrf_ndcg):.4f}")
print(f"MRR:        {sum(rrf_mrr)/len(rrf_mrr):.4f}")
print(f"Recall@100: {sum(rrf_recall)/len(rrf_recall):.4f}")

import pickle
with open("/content/drive/MyDrive/rrf_results_trec-covid.pkl", "wb") as f:
    pickle.dump(rrf_results, f)

print("RRF results saved to Drive")

RRF Hybrid Results — trec-covid
nDCG@10:    0.7068
MRR:        0.9100
Recall@100: 0.1273
RRF results saved to Drive


## Final Comparison

In [17]:
import pickle
from pytrec_eval import RelevanceEvaluator

# Reload saved results
bm25_results = pickle.load(open("/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid/trec-covid_results.pkl", "rb"))
colbert_results = pickle.load(open("/content/drive/MyDrive/colbert_results_trec-covid.pkl", "rb"))
rrf_results = pickle.load(open("/content/drive/MyDrive/rrf_results_trec-covid.pkl", "rb"))

# Rebuild evaluator
qrels_int = {q: {d: int(s) for d, s in docs.items()} for q, docs in qrels.items()}
evaluator = RelevanceEvaluator(qrels_int, {"ndcg_cut.10", "recip_rank.10", "recall.100"})

# BM25
bm25_eval = evaluator.evaluate(bm25_results)
bm25_ndcg = [bm25_eval[q]["ndcg_cut_10"] for q in bm25_eval]
bm25_mrr = [bm25_eval[q]["recip_rank"] for q in bm25_eval]
bm25_recall = [bm25_eval[q]["recall_100"] for q in bm25_eval]

# ColBERT
colbert_eval = evaluator.evaluate(colbert_results)
colbert_ndcg = [colbert_eval[q]["ndcg_cut_10"] for q in colbert_eval]
colbert_mrr = [colbert_eval[q]["recip_rank"] for q in colbert_eval]
colbert_recall = [colbert_eval[q]["recall_100"] for q in colbert_eval]

# RRF
rrf_eval = evaluator.evaluate(rrf_results)
rrf_ndcg = [rrf_eval[q]["ndcg_cut_10"] for q in rrf_eval]
rrf_mrr = [rrf_eval[q]["recip_rank"] for q in rrf_eval]
rrf_recall = [rrf_eval[q]["recall_100"] for q in rrf_eval]

print("All metric summaries rebuilt")

All metric summaries rebuilt


In [18]:
print()
print("=" * 62)
print(f"{'Method':<12} {'nDCG@10':>10} {'MRR':>10} {'Recall@100':>12}")
print("=" * 62)
print(f"{'BM25':<12} {sum(bm25_ndcg)/len(bm25_ndcg):>10.4f} {sum(bm25_mrr)/len(bm25_mrr):>10.4f} {sum(bm25_recall)/len(bm25_recall):>12.4f}")
print(f"{'ColBERT':<12} {sum(colbert_ndcg)/len(colbert_ndcg):>10.4f} {sum(colbert_mrr)/len(colbert_mrr):>10.4f} {sum(colbert_recall)/len(colbert_recall):>12.4f}")
print(f"{'RRF Hybrid':<12} {sum(rrf_ndcg)/len(rrf_ndcg):>10.4f} {sum(rrf_mrr)/len(rrf_mrr):>10.4f} {sum(rrf_recall)/len(rrf_recall):>12.4f}")
print("=" * 62)
print(f"\nSubset: {len(corpus_subset):,} docs | Queries: {len(queries):,} | TOP_K: {TOP_K}")


Method          nDCG@10        MRR   Recall@100
BM25             0.4822     0.6932       0.0927
ColBERT          0.7303     0.9162       0.1371
RRF Hybrid       0.7068     0.9100       0.1273

Subset: 70,480 docs | Queries: 50 | TOP_K: 100


In [20]:
import json
import pickle
from pathlib import Path

# -----------------------------
# CONFIG
# -----------------------------
TOP_N_UI = 10
OUT_PATH = "/content/drive/MyDrive/trec-covid_demo_results.json"

# -----------------------------
# Helpers
# -----------------------------
def safe_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.split())

def make_snippet(doc, max_chars=220):
    title = safe_text(doc.get("title", ""))
    text = safe_text(doc.get("text", ""))
    full = f"{title}. {text}" if title and text else (title or text)
    return full[:max_chars].rstrip() + ("..." if len(full) > max_chars else "")

def build_method_results(method_results, query_id, corpus_lookup, top_n=3):
    ranked_items = list(method_results.get(query_id, {}).items())[:top_n]
    output = []

    for rank, (doc_id, score) in enumerate(ranked_items, start=1):
        doc_id_str = str(doc_id)
        doc = corpus_lookup.get(doc_id) or corpus_lookup.get(doc_id_str) or {}

        output.append({
            "rank": rank,
            "doc_id": doc_id_str,
            "title": safe_text(doc.get("title", "")),
            "score": float(score),
            "snippet": make_snippet(doc)
        })

    return output

# -----------------------------
# Load saved results if needed
# -----------------------------
bm25_path = "/content/drive/MyDrive/IR CW - TREC-COVID/trec-covid/trec-covid_results.pkl"
colbert_path = "/content/drive/MyDrive/colbert_results_trec-covid.pkl"
rrf_path = "/content/drive/MyDrive/rrf_results_trec-covid.pkl"

bm25_results = pickle.load(open(bm25_path, "rb"))
colbert_results = pickle.load(open(colbert_path, "rb"))
rrf_results = pickle.load(open(rrf_path, "rb"))

# -----------------------------
# Build demo JSON
# -----------------------------
# Assumes these already exist in memory from your notebook:
# corpus_subset, queries
# If not, rebuild them before running this cell.

demo_data = {}

for query_id, query_text in queries.items():
    demo_data[query_text] = {
        "query_id": str(query_id),
        "bm25": build_method_results(bm25_results, query_id, corpus_subset, top_n=TOP_N_UI),
        "colbert": build_method_results(colbert_results, query_id, corpus_subset, top_n=TOP_N_UI),
        "rrf": build_method_results(rrf_results, query_id, corpus_subset, top_n=TOP_N_UI),
    }

# -----------------------------
# Save
# -----------------------------
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(demo_data, f, indent=2, ensure_ascii=False)

print(f"Saved demo JSON to: {OUT_PATH}")
print(f"Total queries saved: {len(demo_data):,}")

# Show one sample
first_query = next(iter(demo_data))
print("\nSample query:")
print(first_query)
print("\nSample BM25 result:")
print(json.dumps(demo_data[first_query]["bm25"][:1], indent=2, ensure_ascii=False))

Saved demo JSON to: /content/drive/MyDrive/trec-covid_demo_results.json
Total queries saved: 50

Sample query:
what is the origin of COVID-19

Sample BM25 result:
[
  {
    "rank": 1,
    "doc_id": "dv9m19yk",
    "title": "[What is the origin of SARS-CoV-2?]",
    "score": 16.99302,
    "snippet": "[What is the origin of SARS-CoV-2?]. Every time a pandemic occurs, dozens of theories emerge to attribute the origin of the event to different facts. The COVID-19 pandemic that has hit virtually all the globe has been no..."
  }
]
